# 02 — Baseline Comparison

Every method occupies the same MIGRES retrieval slot. The baseline receives MIGRES-generated queries; SAR alone receives the full state (question, gaps, evidence) and applies its state-aware score. The common MIGRES filter is applied after retrieval, so both raw retrieval metrics and post-filter downstream metrics must be interpreted separately.

Included systems: SAR, BM25, MMR, DPR, Contriever, SPLADE, ColBERTv2, Sentence Transformer Search, and optional InfoGain-RAG when its compatible published checkpoint is configured. Only SAR and Sentence Transformer Search construct the project Sentence Transformer.

In [ ]:
from pathlib import Path
import sys, json, yaml, platform
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
print("Project root:", ROOT)
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("Selected device:", DEVICE)
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    DEVICE = "cpu"
    print("Torch/GPU probe unavailable:", type(e).__name__, e)
PLAN = yaml.safe_load((ROOT / "configs" / "experiments.yaml").read_text())
BASE_CONFIG = [ROOT / "configs" / "default.yaml", ROOT / "configs" / "gemini.yaml"]


In [ ]:
RUN_EXPERIMENT=False
LIMIT_PER_DATASET=None
METHODS=['sar']+PLAN['baselines']['systems']
FROZEN_PATH=ROOT/'outputs/experiments/01_parameter_selection/selected_config.json'
OUT_ROOT=ROOT/'outputs/experiments/02_baseline_comparison'
print('Methods:',METHODS)


In [ ]:
from sar_migres.utils.config import load_config
from sar_migres.datasets.loader import load_phase_questions
from sar_migres.experiments.registry import baseline_overrides
from sar_migres.experiments.parameter_selection import frozen_setting
from sar_migres.experiments.runner import SARMigresPipeline
from sar_migres.experiments.iterative import answer_metrics_from_run, cost_metrics_from_run
from sar_migres.evaluation.plotting import save_bar_plot
from sar_migres.utils.paths import make_run_id
RUN_DIRS_BY_DATASET={}

base=load_config(*BASE_CONFIG)
from sar_migres.utils.config import validate_canonical_protocol
validate_canonical_protocol(base)
if not RUN_EXPERIMENT:
    print('Preflight only. Final split:', base.data.final_split)
    print('Frozen parameters:', 'present' if FROZEN_PATH.exists() else 'MISSING (required before RUN_EXPERIMENT=True)')
else:
    k,w=frozen_setting(FROZEN_PATH)
    for ds in base.data.datasets:
        RUN_DIRS_BY_DATASET[ds]={}
        print('Dataset:',ds,'final labeled split:',base.data.final_split)
        items=load_phase_questions(ds,base.data.dataset_paths[ds],'final',limit=LIMIT_PER_DATASET,final_split=base.data.final_split,validation_fraction=base.data.validation_fraction,validation_seed=base.data.validation_seed)
        print('N questions:',len(items))
        for method in METHODS:
            if method == 'infogain_rag' and not base.baseline.infogain_checkpoint:
                print('SKIP infogain_rag: baseline.infogain_checkpoint is not configured; the published method requires a trained reranker checkpoint.')
                continue
            if method=='sar':
                ov={'retrieval':{'method':'sar'},'sar':{'k':k},'baseline':{'k':k}}
                if w: ov['sar'].update({'alpha':w.alpha,'beta':w.beta,'gamma':w.gamma,'weighting':'static'})
            else:
                ov=baseline_overrides(method,k=k,run_name=f'{ds}_{method}')
            ov['data']={'dataset':ds,'dataset_path':base.data.dataset_paths[ds],'split':base.data.final_split}
            ov['run']={'name':f'{ds}_{method}','output_dir':str(OUT_ROOT),'resume':False}
            cfg=load_config(*BASE_CONFIG,overrides=ov)
            run_id=make_run_id(f'{ds}_{method}')
            run_dir=OUT_ROOT/method/'runs'/run_id
            pipe=SARMigresPipeline(cfg,run_id=run_id,run_dir=run_dir).prepare()
            pipe.run_batch(items,resume=False)
            RUN_DIRS_BY_DATASET[ds][method]=pipe.run_dir
            print(method,'->',pipe.run_dir)


In [ ]:
# Saved comparison charts and resource summaries (all files persist under the Drive-backed outputs/ directory).
PLOT_ROOT=OUT_ROOT/'plots'
for ds, runs in RUN_DIRS_BY_DATASET.items():
    if not runs: continue
    labels=list(runs)
    answer=[answer_metrics_from_run(path).get('exact_match', float('nan')) for path in runs.values()]
    latency=[cost_metrics_from_run(path)['retrieval_latency_total_s'] for path in runs.values()]
    save_bar_plot(labels,answer,'Exact match','Baseline comparison: '+ds,PLOT_ROOT/f'{ds}_exact_match.png')
    save_bar_plot(labels,latency,'Retrieval seconds','Retrieval latency: '+ds,PLOT_ROOT/f'{ds}_retrieval_seconds.png')
    summary={name:cost_metrics_from_run(path) for name,path in runs.items()}
    (PLOT_ROOT/f'{ds}_cost_summary.json').write_text(json.dumps(summary,indent=2,allow_nan=True),encoding='utf-8')
print('Comparison plots and cost summaries:',PLOT_ROOT)
